In [14]:
# 네이버검색결과화면
import requests
keyword = 'ai'
url = f'https://search.naver.com/search.naver?ssc=tab.news.all&where=news&sm=tab_jum&query={keyword}'
# requests로 url로 요청해서 응답을 가져옴
response = requests.get(url)


In [15]:
import pandas as pd
from bs4 import BeautifulSoup
# 요청이 성공했는지 확인
try:
	# 요청이 성공했는지 확인
	response.raise_for_status()
	print("연결 성공!!")
	# 가져온 결과를 콘솔에 출력
	# print(response.text)

	# 가져온 결과에서 다음을 만족하는 요소들을 가져옴
	# .sds-comps-full-layout a[data-heatmap-target=".tit"]
	soup = BeautifulSoup(response.text, 'lxml')
	links = soup.select('.sds-comps-full-layout a[data-heatmap-target=".tit"]')

	titles = []
	hrefs = []
	# 가져온 요소들을 콘솔에 출력
	for link in links:
		# 제목과 링크를 추출해서 콘솔에 출력
		title = link.text
		href = link.get('href')
		# 제목과 링크를 제목 리스트, 링크 리스트에 각각 뒤에 추가
		titles.append(title)
		hrefs.append(href)

	print(titles)
	print(hrefs)
	# 판다스를 이용하여 DataFrame을 생성
	# 추가할 때 열 이름은 title 과 href로 지정
	df = pd.DataFrame({
		'title' : titles,
		'href' : hrefs
	})
	print(df)

	# csv 파일로 저장
	df.to_csv(f'naver_nl_{keyword}.csv', index=False, encoding='utf-8-sig')

except Exception as e:
	print(f"예외 발생 : {e}")
	
if response.status_code == 200:
	print("연결 성공!")


연결 성공!!
['[단독] 크래프톤·한화, 피지컬 AI 합작사 설립', '크래프톤·한화에어로, 피지컬AI 합작사 추진', '[특징주] 크래프톤, 한화에어로와 피지컬AI 합작사 추진에 강세', '[단독]가상환경서 첨단무기 테스트…크래프톤-한화에어로 ‘AI동맹’', '크래프톤·한화, 합작법인 만든다…방산 피지컬 AI 사업화', '“한국판 안두릴 만들겠다”…크래프톤·한화에어로, 피지컬 AI 협력', '크래프톤-한화에어로스페이스, 피지컬 AI 동맹...합작법인 설립', '크래프톤·한화에어로 ‘피지컬 AI’ 동맹…합작법인 설립', '크래프톤, 한화에어로스페이스와 ‘피지컬 AI’ 협력…JV 설립 추진', 'AI가 산재이력 학습해 위험사업장 선별…노동부 AX 세미나 개최', '"인간보다 성능 52% 향상"…노동부, AI 산재 예측 모델 공개', '노동행정도 AI로…노동부, 산재 예측 모델 첫 공개', "AI가 산재 '상위 0.6%' 선별 예측...임금체불 AI도 개발", '크래프톤이 게임AI 넘어 피지컬AI로 보폭 넓히는 이유', "크래프톤, 한화에어로와 '피지컬AI' 동맹…합작법인도 설립", '한화에어로-크래프톤 ‘피지컬AI’ 동맹 구축한다', '한화에어로, 게임 제작사 크래프톤과 ‘피지컬 AI’ 개발한다', "한화에어로-크래프톤, 피지컬 AI 공동개발 '맞손'", '"SK하이닉스, AI 메모리 수요 급증 최대 수혜…목표가 170만원"-KB', 'SK하이닉스, 목표가 170만원 상향...AI 메모리 수요 급증 수혜-KB', '삼성SDI, 피지컬AI용 전고체 배터리 첫 공개', '한컴, 오픈데이터로더 PDF v2.0 공개…문서 AI 시장 공략 박차', '李대통령 "국민주권 정부니까 가능…AI전환 신속 추진"', '과기정통·산업·중기부, 4320억 규모 AI전환 사업 통합 공고', 'NC AI, 중소·인디 게임 개발사 지원 생태계 구축', "NC AI, '바르코 게임 AI' 모바일 게임 제작 생태계 구축", 'NC AI, ‘바르코 AI’ 기반 모바일 게임 제작 생태계 만든다

In [ ]:
# 위 예제를 이용하여 키워드가 주어졌을 때 뉴스 목록을 가져와서 데이터프레임으로 반환하는
# 함수를 만들고 테스트 하세요
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def get_naver_news_list(keyword : str, start : int=1):

	url = f'https://search.naver.com/search.naver'

	params = {
		'where' : 'news',
		'query' : keyword,
		'start' : start,
	}
	headers = {
		'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36',
		'Referer' : 'https://www.naver.com/'
	}
	# 뉴스 검색 화면을 요청
	response = requests.get(url, params=params, headers=headers)
	
	try:
		# 연결
		response.raise_for_status()
		
		# Beautifulsoup 객체 생성
		soup = BeautifulSoup(response.text, 'lxml')

		# 뉴스 박스들
		links_divs = soup.select('.n8_DDlCjHxLYRO50CY50')

		hrefs, titles = [], []
		# 반복문으로 제목과 링크를 추출하여 리스트에 담음
		for link_div in links_divs:
			# 기사 제목링크 => 여기에서 기사 제목 추출
			title_link = link_div.select('.XEtVZ4N7DI2Pv29xe7f3')
			# 실제 기사 네이버링크 => 여기에서 기사 링크 추출
			href_link = link_div.select('.RQNKk8QZaQZble3gmgEj [data-heatmap-target=".nav"]')
			# 리스트에 제목과 링크를 추가
			hrefs.append(href_link.get("href"))
			titles.append(title_link.text)

		return pd.DataFrame({
			'title' : titles,
			'href' : hrefs,
		})
	except Exception as e:
		print(f"예외 발생 : {e}")
		return None
	


In [16]:
# 뉴스 리트스 DataFrame을 csv로 저장
def save_nl(df:pd.DataFrame,keyword:str, start:int = 1):
	if df is None or df.empty:
		return
	df.to_csv(f'naver_nl_{keyword}_{start//10+1}.csv', index=False, encoding='utf-8-sig')


In [17]:
# start = 41
keyword = 'ai'
# nl_df = get_naver_news_list(keyword,start)
# save_nl(nl_df, keyword, start)
#print(nl_df)

# 반복문으로 총 50개의 ai 뉴스 목록을 가져오도록 작업을 하세요
# 단, 작업할 때 time.sleep(2)를 이용하여
for page in range(1,5+1):
	start = (page - 1) * 10 + 1
	nl_df = get_naver_news_list(keyword,start)
	save_nl(nl_df, keyword, start)
	print(f'{keyword}{page}페이지 완료')
	time.sleep(2)
	
	

ai1페이지 완료
ai2페이지 완료
ai3페이지 완료
ai4페이지 완료
ai5페이지 완료
